### 1 : Importing Libraries

In [1]:
# ignore the warnings in the output
import warnings
warnings.filterwarnings("ignore")

In [2]:
# import libraries and packages
# from scripts import utils
import numpy as np
import pandas as pd 
import utils
from constants import *
import matplotlib.pyplot as plt
import mlflow
from pycaret.classification import *
import os
import sqlite3
from sqlite3 import Error
from pycaret.regression import *

 ### 2 Reading Data

In [3]:
##Copy the cleaned data obtained from the data_cleaning notebook into the Data folder before proceeding
%time
df = pd.read_csv('Data/cleaned_data.csv')

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 8.11 µs


In [4]:
utils.read_db_file(DB_FILE_NAME_EXP)

[('experiments',), ('params',), ('alembic_version',), ('experiment_tags',), ('tags',), ('registered_models',), ('runs',), ('registered_model_tags',), ('model_version_tags',), ('model_versions',), ('latest_metrics',), ('metrics',)]


In [5]:
df.head()

,created_date,city_tier,first_platform_c,first_utm_medium_c,first_utm_source_c,total_leads_droppped,referred_lead,app_complete_flag,assistance_interaction,career_interaction,payment_interaction,social_interaction,syllabus_interaction
0,2021-07-01 00:08:15,1.0,Level0,Level11,Level2,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
1,2021-07-01 00:16:43,2.0,Level3,Level0,others,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
2,2021-07-01 00:22:20,1.0,Level3,Level0,Level0,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
3,2021-07-01 00:23:13,1.0,Level1,Level3,others,2.0,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2021-07-01 00:28:38,1.0,Level3,Level0,Level0,1.0,0.0,0,0.0,0.0,0.0,0.0,0.0


In [6]:
# drop the data column as it is not needed for training
df = df.drop(['created_date'], axis=1)

In [7]:
df.head()

,city_tier,first_platform_c,first_utm_medium_c,first_utm_source_c,total_leads_droppped,referred_lead,app_complete_flag,assistance_interaction,career_interaction,payment_interaction,social_interaction,syllabus_interaction
0,1.0,Level0,Level11,Level2,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
1,2.0,Level3,Level0,others,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
2,1.0,Level3,Level0,Level0,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
3,1.0,Level1,Level3,others,2.0,0.0,0,0.0,0.0,0.0,0.0,0.0
4,1.0,Level3,Level0,Level0,1.0,0.0,0,0.0,0.0,0.0,0.0,0.0


### 3 Setting up Environment: 

In [8]:
# create a connection and setup a SQLite database with the name "lead_scoring_model_experimentation.db" in 
# 'Assignment/02_training_pipeline/notebooks/' location
DB_PATH = "Assignment/02_training_pipeline/notebooks/"
DB_FILE_NAME = "lead_scoring_model_experimentation.db"

# utils.build_dbs(DB_PATH,r"lead_scoring_model_experimentation.db")
conn = None
# opening the connection for creating the sqlite db
try:
    #sqlite3 "lead_scoring_model_experimentation.db"
    conn = sqlite3.connect(DB_PATH+DB_FILE_NAME)
    print(sqlite3.version)
except Error as e:
    print(e)
#    sqlite3(DB_FILE_NAME)
#    sqlite "lead_scoring_model_experimentation.db"
#    sqlite3.create lead_scoring_model_experimentation.db
finally:
    if conn:
        conn.close()
        

unable to open database file


In [9]:
'''
Now you need to start the MLflow server in a new terminal. 
Note: Before you start the MLflow server, create a folder named mlruns in the assignment directory.
Now you need to run the command to start MLflow server such that:
1. The lead_scoring_model_experimentation.db which you created above is used as the backend-store.
2. mlruns folder is used as an artifact directory. 
3. The server runs on the port 6006.

The steps to do so are as follows:
Open a new terminal.
Then go to the Assignment directory using the cd command. Type the command: cd Assignment/
Create a folder named mlruns here. You can create this folder using either the command line or GUI. To create this folder via the command line run the command: mkdir ./mlruns
Then, type the following command to start the MLflow server: 
mlflow server --backend-store-uri='sqlite:///./02_training_pipeline/notebooks/lead_scoring_model_experimentation.db' --default-artifact-root="./mlruns" --port=6006 --host=0.0.0.0
'''

'\nNow you need to start the MLflow server in a new terminal. \nNote: Before you start the MLflow server, create a folder named mlruns in the assignment directory.\nNow you need to run the command to start MLflow server such that:\n1. The lead_scoring_model_experimentation.db which you created above is used as the backend-store.\n2. mlruns folder is used as an artifact directory. \n3. The server runs on the port 6006.\n\nThe steps to do so are as follows:\nOpen a new terminal.\nThen go to the Assignment directory using the cd command. Type the command: cd Assignment/\nCreate a folder named mlruns here. You can create this folder using either the command line or GUI. To create this folder via the command line run the command: mkdir ./mlruns\nThen, type the following command to start the MLflow server: \nmlflow server --backend-store-uri=\'sqlite:///./02_training_pipeline/notebooks/lead_scoring_model_experimentation.db\' --default-artifact-root="./mlruns" --port=6006 --host=0.0.0.0\n'

In [10]:
# Once, your server is successfully running, create a mlflow tracking uri at "http://0.0.0.0:6006"
mlflow.set_tracking_uri("http://0.0.0.0:6006")

In [11]:
#utils.get_trained_model()

In [12]:
# setup pycaret 
!pip install pycaret

### 4 : Model Experimentation with pycaret

In [13]:
# create a experimentation with pycaret and exclude ['gbc','knn','qda', 'dummy', 'svm', 'ada']

data_unseen = df.sample(frac=0.1, random_state=42)      # Sample 10% of the data to become the unseen test set
df = df.drop(data_unseen.index)                       # Use the remaining 90% as the training (& validation) data

df.reset_index(drop=True, inplace=True)
data_unseen.reset_index(drop=True, inplace=True)

print('Data for Model Training & Validation: ' + str(df.shape))
print('Unseen Data For Predictions: ' + str(data_unseen.shape))

Data for Model Training & Validation: (215068, 12)
Unseen Data For Predictions: (23896, 12)


In [14]:
df.head()

,city_tier,first_platform_c,first_utm_medium_c,first_utm_source_c,total_leads_droppped,referred_lead,app_complete_flag,assistance_interaction,career_interaction,payment_interaction,social_interaction,syllabus_interaction
0,1.0,Level0,Level11,Level2,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
1,2.0,Level3,Level0,others,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
2,1.0,Level3,Level0,Level0,1.0,0.0,1,0.0,0.0,0.0,0.0,0.0
3,1.0,Level1,Level3,others,2.0,0.0,0,0.0,0.0,0.0,0.0,0.0
4,1.0,Level3,Level0,Level0,1.0,0.0,0,0.0,0.0,0.0,0.0,0.0


In [15]:
from pycaret.regression import *

In [16]:
#expt_basic = setup(
#    data = df, 
#    target = 'app_complete_flag', 
#    session_id=42,                      # Random seed to ensure reproducibility of the experiment with the same data
#    train_size=0.8,                     # 80% training data & 20% held-out validation data
#    ignore_features=[],
#    numeric_imputation="median",        # "mean" by default
#    categorical_imputation="mode",      # "constant" (not_available) by default
#)

Baseline_model_exp01 = setup(
    data = df, target = 'app_complete_flag',
    session_id = 42, #fix_imbalance=False, #ignore_feature=[''],
    n_jobs=-1,use_gpu=True,
    log_experiment=True, experiment_name='Baseline_model_exp01',
    log_plots=True, log_data=True,
    silent=True, verbose=True,
    log_profile=False
)

,Description,Value
0,session_id,42
1,Target,app_complete_flag
2,Original Data,"(215068, 12)"
3,Missing Values,False
4,Numeric Features,7
5,Categorical Features,4
6,Ordinal Features,False
7,High Cardinality Features,False
8,High Cardinality Method,None
9,Transformed Train Set,"(150547, 43)"


2024/02/14 13:53:17 INFO mlflow.tracking.fluent: Experiment with name 'Baseline_model_exp01' does not exist. Creating a new experiment.


In [17]:
models()

,Name,Reference,Turbo
ID,,,
lr,Linear Regression,cuml.linear_model.linear_regression.LinearRegr...,True
lasso,Lasso Regression,cuml.linear_model.lasso.Lasso,True
ridge,Ridge Regression,cuml.linear_model.ridge.Ridge,True
en,Elastic Net,cuml.linear_model.elastic_net.ElasticNet,True
lar,Least Angle Regression,sklearn.linear_model._least_angle.Lars,True
llar,Lasso Least Angle Regression,sklearn.linear_model._least_angle.LassoLars,True
omp,Orthogonal Matching Pursuit,sklearn.linear_model._omp.OrthogonalMatchingPu...,True
br,Bayesian Ridge,sklearn.linear_model._bayes.BayesianRidge,True
ard,Automatic Relevance Determination,sklearn.linear_model._bayes.ARDRegression,False


In [ ]:
best = compare_models(sort="RMSE", exclude=['knn', 'dummy', 'svm', 'ada'], fold=5)

IntProgress(value=0, description='Processing: ', max=79)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,13:53:35
Status,. . . . . . . . . . . . . . . . . .,Creating Logs
Estimator,. . . . . . . . . . . . . . . . . .,Light Gradient Boosting Machine


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.3368,0.1678,0.4097,0.3286,0.2873,0.3352,4.0100
xgboost,Extreme Gradient Boosting,0.3354,0.1682,0.4101,0.3273,0.2874,0.3341,0.5080
rf,Random Forest Regressor,0.3350,0.1687,0.4108,0.3249,0.2878,0.3333,3.1480
et,Extra Trees Regressor,0.3338,0.1700,0.4123,0.3201,0.2887,0.3322,5.6480
dt,Decision Tree Regressor,0.3338,0.1703,0.4127,0.3185,0.2890,0.3322,0.2980
gbr,Gradient Boosting Regressor,0.3525,0.1720,0.4147,0.3118,0.2909,0.3510,9.5720
br,Bayesian Ridge,0.3891,0.1925,0.4387,0.2299,0.3060,0.3904,0.3220
ridge,Ridge Regression,0.3890,0.1925,0.4388,0.2299,0.3060,0.3903,0.0300
lr,Linear Regression,0.3917,0.1931,0.4394,0.2276,0.3066,0.3925,0.1860
huber,Huber Regressor,0.3611,0.2057,0.4535,0.1771,0.3141,0.3204,5.7860


findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial, Liberation Sans, Bitstream Vera Sans, sans-serif
findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial, Liberation Sans, Bitstream Vera Sans, sans-serif
findfont: Font family ['sans-serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial, Liberation Sans, Bitstream Vera Sans, sans-serif


In [ ]:
# create a model which gives the highest accuracy
lgbm = create_model("lightgbm", fold=10)

In [ ]:
tuned_lgbm = tune_model(lgbm, fold=5, optimize="RMSE")

In [ ]:
print(lgbm, "\n")
print(tuned_lgbm)

In [ ]:
# create feature importance plot
plot_model(tuned_lgbm, plot="feature")  

### 5 : Model Experimentation after dropping features

From the above feature tests we can claerly see that some of the features are not significant. We will now drop all the insignificant features and select only the significant ones.
The list of the significant features is 
['total_leads_droppped', 'city_tier', 'referred_lead', 'app_complete_flag', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c'].
So now you will train your model with onlly these features.

Also note that in our previous experiments we saw that tree based models are our top performers. In case of tree based models we do not require transformationss normalization, scaling etc. So make sure that you use setup pycaret in the proper way. i.e, make sure that you use normalize = False and transformation = False.

In [ ]:
utils.build_dbs()

In [ ]:
utils.encode_features()

In [ ]:
utils.read_db_file(DB_FILE_NAME_NOTE)

In [ ]:
utils.read_table_data(FEAT_TAB_NAME)

In [ ]:
utils.read_table_data(TARG_TAB_NAME)

In [ ]:
# 
# ['total_leads_droppped', 'city_tier', 'referred_lead', 'app_complete_flag', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c']
#
# Train the model using the features listed above. Since we are using tree models we do not require any transformaions 
# such as normalization, scaling etc.So make sure that you use setup pycaret in the proper way. i.e, make sure that you use 
# normalize = False and transformation = False.

con = sqlite3.connect('/home/Assignment/02_training_pipeline/notebooks/lead_scoring_data_cleaning.db')
FEATDATA_FRAME = pd.read_sql("""SELECT * from features""", con)

#expt_basic = setup(
#    data = FEATDATA_FRAME, 
#    target = 'app_complete_flag', 
#    session_id=42,                      # Random seed to ensure reproducibility of the experiment with the same data
#    train_size=0.8,                     # 80% training data & 20% held-out validation data
#    ignore_features=[],
#    numeric_imputation="median",        # "mean" by default
#    categorical_imputation="mode",      # "constant" (not_available) by default
#)

Baseline_model_exp00 = setup(
    data = FEATDATA_FRAME, target = 'app_complete_flag',
    session_id = 42, fix_imbalance=False, #ignore_feature=[''],
    n_jobs=-1,use_gpu=True,
    log_experiment=True, experiment_name='Baseline_model_exp00',
    log_plots=True, log_data=True,
    silent=True, verbose=True,
    log_profile=False
)

In [ ]:
models()

In [ ]:
# create a experimentation with pycaret and exclude ['gbc','knn','qda', 'dummy', 'svm', 'ada']. 
best = compare_models(sort="RMSE", exclude=['knn', 'dummy', 'svm', 'ada'], fold=5)

In [ ]:
# You should get lightgbm as the best performing model. So now we will train a lightGBM model manually using pycaret
lgbm = create_model("lightgbm", fold=10)

In [ ]:
# Tune the hyper parameters of the lightgbm model using optuna on 10 folds and optimise AUC as that was our system metric, 
# hence we will optimise AUC
tuned_lgbm = tune_model(lgbm, fold=5, optimize="RMSE")

In [ ]:
# Print the final models configuration so that we can use it in the model retraining pipeline 
print(lgbm, "\n")
print(tuned_lgbm)